# Preprocessing Raw Data

In [1]:
import pandas as pd
import os
import io
import numpy as np
import math
import sys
from pathlib import Path
from scipy.optimize import curve_fit
pd.reset_option("display.precision")
pd.reset_option("display.float_format")
pd.set_option("display.max_columns", None)

### Changing to grandparent directory

In [2]:
# Get the absolute path of the current file
current_file_path = Path("").resolve()
grandparent_dir = current_file_path.parent.parent

# Append this directory to the system path
sys.path.append(str(grandparent_dir))

In [3]:
folder_path = os.path.join(grandparent_dir,"Raw Data - FeCN platinum 8")

## 1. Raw File Import

Importing experimental .dta files in `Raw Data - FeCN platinum 8` folder for all potential modulations. Files are named by RMS Voltage, not peak voltage.

In [4]:
def update_units(unit_dict,cols,new_units, df = None):
    """
    Adds column names and associated units to units dictionary.
    
    Args:
        unit_dict (dict): dictionary containing the columns and their units
        cols (list): list of individual or listed column names as strings
        new_units: list of individual or listed values as strings corresponding to the cols variable
        df (pd.DataFrame): dataframe that unit_dict corresponds to that helps to reorder 
        unit_dict key in order of DataFrame's columns
    Returns:
        None
    """
    for col, units in zip(cols,new_units):
        if type(col) != list:
            unit_dict[col] = units
        else:
            unit_dict.update(zip(col,units))
    if df is not None:
        temp_dict = {c: unit_dict[c] for c in df.columns if c in unit_dict}
        unit_dict.clear()        # remove old keys
        unit_dict.update(temp_dict)  # insert in df column order
    return

In [5]:
def save_visualize_units(units_dict, filepath, disp = False):
    """
    Saves and visualizes units dictionary as a vertical pandas DataFrame.
    
    Args:
        unit_dict (dict): dictionary containing the columns and their units
        filepath (str): filepath to save unit dictionary to
        disp (bool): flag for displaying unit dictionary as DataFrame
        
    Returns:
        None
    """

    df = pd.DataFrame(list(units_dict.items()), columns=['Unit', 'Value'])
    if disp == True:
        with pd.option_context('display.max_rows', None):
            display(df)
    df.to_csv(filepath,index=False)
    return

In [6]:
#Initalizations
dfs = {}
num_currents = 10
V_I_cols = [col
          for x in range(1, num_currents + 1)
          for col in (f"IHr{x}", f"IHi{x}", f"VHr{x}", f"VHi{x}")] #IHrj, IHij, VHrj, VHij from j = 1 to j = 10
V_I_units = [col
          for x in range(1, num_currents + 1)
          for col in ("A (raw)", "A (raw)", "V (raw)", "V (raw)")] #units for #IHrj, IHij, VHrj, VHij
keep_cols = ['Freq','Vmod'] + V_I_cols #columns to extract from raw files for analysis
units = {'Freq': "Hz",
         'Vmod': "V"}
update_units(units,[V_I_cols,"Modulation (RMS)","File Number","Rank"],[V_I_units,"V_RMS Label","None","None"])

# Iterate over EISPOT files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith('.DTA') and filename.startswith("EISPOT"):
        file_path = os.path.join(folder_path, filename)
        
        # Open and read the file content, skipping instrument settings up to "ZCURVE"
        with open(file_path, "r", encoding="ISO-8859-1") as f:
            content = f.readlines()
            for i,line in enumerate(content):
                if line.__contains__("ZCURVE"):
                    index = i + 1 #finds line where datatable resides (below "Zcurve" label)
                    break
            content = content[index:]
                
            # Read the remaining lines and create a DataFrame
            df = pd.DataFrame([line.split('\t') for line in content if line.strip()])
            df = df.map(lambda x: x.strip() if isinstance(x, str) else x) # Remove newline characters from the last column
            df = df.drop(0, axis=1) #drops first column
            df.columns = df.iloc[0] # Sets the header names to values from the first row
            df = df.drop(0) # Drop the first row (since it's now used as header)
            df = df.iloc[1:] # Drops row with units
            df.dropna(how='all',inplace=True)
            df["Freq"] = df["Freq"].astype(float)
            df.sort_values(by = ["Freq"],ascending=False,inplace=True) #sorts in decreasing Frequency order (in case its not)
            df.reset_index(drop=True,inplace=True)
            df = df[keep_cols]
            
            #Saves Files to Output Folder
            filename_stripped = filename[:-4]
            dfs[filename_stripped] = df #creates list of dataframes made from each run & potential modulation file

#Generating a Master Table with Identifying Potential Modulation, File Number, and Rank Row Labels
keys = dfs.keys()
dfs_list = []
for key in keys:
    names = key.split("_")
    modulation = names[-2]
    file_num = names[-1]
    df_temp = dfs[key]
    df_temp["Modulation (RMS)"] = modulation #RMS Voltage (following naming convention of files)
    df_temp["File Number"] = file_num
    df_temp["Rank"] = pd.Series(range(len(df_temp), 0, -1)) #ranks each frequency in increasing order from 1 (smallest omega) to 73 (largest omega) 
    dfs_list.append(df_temp)
df_all = pd.concat(dfs_list,axis=0) #concatenates all potential modulations with the generated row labels
df_all.head()

,Freq,Vmod,IHr1,IHi1,VHr1,VHi1,IHr2,IHi2,VHr2,VHi2,IHr3,IHi3,VHr3,VHi3,IHr4,IHi4,VHr4,VHi4,IHr5,IHi5,VHr5,VHi5,IHr6,IHi6,VHr6,VHi6,IHr7,IHi7,VHr7,VHi7,IHr8,IHi8,VHr8,VHi8,IHr9,IHi9,VHr9,VHi9,IHr10,IHi10,VHr10,VHi10,Modulation (RMS),File Number,Rank
0,1500059.0,0.0100416,-0.1554099,0.1005158,-0.5449082,0.7273971,0.0001771,0.0003456,0.0022918,0.0003195,0.0001909,2.13747E-005,0.0004798,-0.0005833,-1.733797E-005,-1.892305E-005,-0.000109,-0.0001465,-2.630125E-005,-2.461887E-005,3.031921E-005,-8.406351E-005,-1.053838E-005,-3.035442E-005,-0.0001021,-0.0003127,-2.854248E-005,-3.008579E-005,9.064237E-005,-7.600245E-005,-1.538056E-005,1.34838E-005,-8.371008E-005,-0.0001187,4.489208E-005,-2.873351E-005,2.263952E-005,-6.906456E-005,3.511109E-005,2.369168E-005,5.62612E-005,0.0001209,10mV,2,73
1,1191504.0,0.0127297,-0.1884086,-0.1575368,-1.05014,-0.4740141,-0.0005022,-0.0004305,-0.0034932,-0.0001822,-0.0001355,-9.377198E-005,-0.0005188,0.0002364,-3.561616E-005,-2.267352E-007,-0.0001384,-1.318532E-005,-2.174279E-005,-2.626035E-005,-0.0003523,0.0001501,-2.981224E-005,1.589712E-005,-0.0001955,6.773416E-005,9.188901E-006,-3.798175E-005,-0.0002933,-4.469837E-005,5.080743E-006,1.137075E-005,-0.0001775,3.111432E-005,9.50952E-006,3.339876E-006,7.893879E-005,0.0002979,-3.545116E-005,6.209761E-006,-0.000311,4.340522E-005,10mV,2,72
2,946464.9,0.0125644,-0.2247906,-0.1077484,-1.107873,-0.2566108,-0.0004818,-0.0002192,-0.0025545,0.0002215,-0.0001282,0.0001399,-0.0003222,0.0005156,1.005217E-005,-3.252645E-005,0.0001972,-0.000221,-3.008847E-005,1.714582E-005,-0.0002009,0.000115,-9.3875E-006,2.866518E-005,0.0001298,-3.686641E-005,-2.070225E-005,3.939122E-005,-0.0000951,0.0002931,-9.987737E-006,7.646624E-006,-0.0001524,5.3528E-005,-3.220413E-006,8.119736E-006,-0.000218,0.000129,-3.121723E-005,-2.413802E-005,-0.0003514,1.44979E-005,10mV,2,71
3,751816.4,0.0125862,-0.2421683,-0.0776115,-1.130004,-0.1442501,-0.000378,-8.300774E-005,-0.0017554,0.0001786,-6.02461E-005,8.408003E-005,-0.0001097,-4.614006E-005,2.129271E-005,2.460842E-005,0.0001312,0.0001698,1.643202E-006,-1.570839E-005,-8.88249E-005,7.085875E-005,-1.629349E-005,-3.153985E-005,-0.0001818,-0.0001072,-1.555102E-005,-2.120703E-005,-0.0001035,-4.086062E-005,-4.300034E-006,2.475572E-006,0.0001769,-0.0001566,7.459836E-006,3.618654E-005,0.0001038,0.0001558,-2.017478E-007,-5.072624E-006,0.0000988,0.0002441,10mV,2,70
4,597246.1,0.012618,-0.1980543,-0.1649427,-0.981439,-0.5840084,-0.0001852,-0.0002086,-0.0008408,-0.0006683,-9.225496E-005,-0.0001134,-0.000529,-0.0005786,2.091425E-005,-3.783447E-005,-0.0001304,-0.0002228,-2.214755E-005,2.339273E-005,-4.814566E-005,-0.0001464,1.862971E-005,2.310216E-005,1.890911E-005,0.0001798,-1.425622E-006,-1.560117E-005,1.21491E-005,7.170206E-005,-1.143944E-005,-1.896254E-005,0.0001144,-4.286413E-005,-1.001463E-005,2.083322E-005,-0.0002315,5.342066E-005,4.235655E-006,-8.113683E-006,0.0001194,0.0001447,10mV,2,69


#### The units of `Frequency` (Freq) is Hz, `Voltage` (Vmod, VHi, VHr) is Volts, and `Current` (IHi, IHr) is Amps, unless otherwise specified in the column name.

# 2. Instrument Scaling Factor

Correction factor applied by the Gamry Instrument is re-corrected.

In [7]:
#Instrument Scaling Factor
N = 128 #number of frequencies recorded for each modulation amplitude
mod = 2/N #correction factor (peak current and peak voltage)
mod_cols = ["IHr","IHi","VHr","VHi"]
nums = range(1, 11)
for num in nums:
    for col in mod_cols:
        cname = f"{col}{num}"
        df_all[cname] = df_all[cname].astype(float) * mod #multiplies each current and voltage by mod

#Save Result  
save_path = os.path.join(current_file_path,"unprocessed_data_modFactor.csv")
df_all.to_csv(save_path,index=False)

#Update Units
V_I_units = [col
          for x in range(1, num_currents + 1)
          for col in ("A (I_peak)", "A (I_peak)", "V (V_peak)", "V (V_peak)")]
update_units(units,[V_I_cols],[V_I_units])
save_units = os.path.join(current_file_path,"unprocessed_data_modFactor_units.csv")
save_visualize_units(units, save_units, disp = False)

df_all.head()

,Freq,Vmod,IHr1,IHi1,VHr1,VHi1,IHr2,IHi2,VHr2,VHi2,IHr3,IHi3,VHr3,VHi3,IHr4,IHi4,VHr4,VHi4,IHr5,IHi5,VHr5,VHi5,IHr6,IHi6,VHr6,VHi6,IHr7,IHi7,VHr7,VHi7,IHr8,IHi8,VHr8,VHi8,IHr9,IHi9,VHr9,VHi9,IHr10,IHi10,VHr10,VHi10,Modulation (RMS),File Number,Rank
0,1500059.0,0.0100416,-0.002428,0.001571,-0.008514,0.011366,0.000003,0.000005,0.000036,0.000005,2.982813e-06,3.339797e-07,0.000007,-9.114063e-06,-2.709058e-07,-2.956727e-07,-0.000002,-2.289063e-06,-4.109570e-07,-3.846698e-07,4.737377e-07,-0.000001,-1.646622e-07,-4.742878e-07,-1.595313e-06,-4.885938e-06,-4.459762e-07,-4.700905e-07,1.416287e-06,-1.187538e-06,-2.403213e-07,2.106844e-07,-0.000001,-1.854687e-06,7.014388e-07,-4.489611e-07,3.537425e-07,-1.079134e-06,5.486108e-07,3.701825e-07,8.790813e-07,1.889062e-06,10mV,2,73
1,1191504.0,0.0127297,-0.002944,-0.002462,-0.016408,-0.007406,-0.000008,-0.000007,-0.000055,-0.000003,-2.117187e-06,-1.465187e-06,-0.000008,3.693750e-06,-5.565025e-07,-3.542738e-09,-0.000002,-2.060206e-07,-3.397311e-07,-4.103180e-07,-5.504687e-06,0.000002,-4.658162e-07,2.483925e-07,-3.054688e-06,1.058346e-06,1.435766e-07,-5.934648e-07,-4.582812e-06,-6.984120e-07,7.938661e-08,1.776680e-07,-0.000003,4.861613e-07,1.485863e-07,5.218556e-08,1.233419e-06,4.654687e-06,-5.539244e-07,9.702752e-08,-4.859375e-06,6.782066e-07,10mV,2,72
2,946464.9,0.0125644,-0.003512,-0.001684,-0.017311,-0.004010,-0.000008,-0.000003,-0.000040,0.000003,-2.003125e-06,2.185938e-06,-0.000005,8.056250e-06,1.570652e-07,-5.082258e-07,0.000003,-3.453125e-06,-4.701323e-07,2.679034e-07,-3.139063e-06,0.000002,-1.466797e-07,4.478934e-07,2.028125e-06,-5.760377e-07,-3.234727e-07,6.154878e-07,-1.485937e-06,4.579688e-06,-1.560584e-07,1.194785e-07,-0.000002,8.363750e-07,-5.031895e-08,1.268709e-07,-3.406250e-06,2.015625e-06,-4.877692e-07,-3.771566e-07,-5.490625e-06,2.265297e-07,10mV,2,71
3,751816.4,0.0125862,-0.003784,-0.001213,-0.017656,-0.002254,-0.000006,-0.000001,-0.000027,0.000003,-9.413453e-07,1.313750e-06,-0.000002,-7.209384e-07,3.326986e-07,3.845066e-07,0.000002,2.653125e-06,2.567503e-08,-2.454436e-07,-1.387889e-06,0.000001,-2.545858e-07,-4.928102e-07,-2.840625e-06,-1.675000e-06,-2.429847e-07,-3.313598e-07,-1.617187e-06,-6.384472e-07,-6.718803e-08,3.868081e-08,0.000003,-2.446875e-06,1.165599e-07,5.654147e-07,1.621875e-06,2.434375e-06,-3.152309e-09,-7.925975e-08,1.543750e-06,3.814062e-06,10mV,2,70
4,597246.1,0.012618,-0.003095,-0.002577,-0.015335,-0.009125,-0.000003,-0.000003,-0.000013,-0.000010,-1.441484e-06,-1.771875e-06,-0.000008,-9.040625e-06,3.267852e-07,-5.911636e-07,-0.000002,-3.481250e-06,-3.460555e-07,3.655114e-07,-7.522759e-07,-0.000002,2.910892e-07,3.609713e-07,2.954548e-07,2.809375e-06,-2.227534e-08,-2.437683e-07,1.898297e-07,1.120345e-06,-1.787413e-07,-2.962897e-07,0.000002,-6.697520e-07,-1.564786e-07,3.255191e-07,-3.617187e-06,8.346978e-07,6.618211e-08,-1.267763e-07,1.865625e-06,2.260937e-06,10mV,2,69


# 3. Rotation

Rotating I1 and I2 vectors by the angle of rotation found to make VHi1 ≈ 0.

In [8]:
#Calculate Theta
df_all["theta"] = -np.arctan2(df_all["VHi1"],df_all["VHr1"]) #arctan2 returns a theta between -180 and 180 degrees, representing the angle from the positive x-axis

#Updating Harmonic Voltage and Current Columns (VH,IH) to Rotated Values
nums = range(1, 11)
real_cols = ["IHr","VHr"]
imag_cols = ["IHi","VHi"]
for num in nums:
    for real,imag in zip(real_cols,imag_cols):
        rname = f"{real}{num}"
        iname = f"{imag}{num}"
        r_orig = df_all[rname].astype(float).copy()
        i_orig = df_all[iname].astype(float).copy()
        df_all[rname] = np.cos(num * df_all["theta"]) * r_orig - np.sin(num * df_all["theta"]) * i_orig
        df_all[iname] = np.sin(num * df_all["theta"]) * r_orig + np.cos(num * df_all["theta"]) * i_orig

#Saving Rotated Currents/Voltages/Admittances as Separate Columns
A1 = (df_all["IHr1"] + df_all["IHi1"]*1j) / (df_all["VHr1"] + df_all["VHi1"]*1j)
A2 = (df_all["IHr2"] + df_all["IHi2"]*1j) / (df_all["VHr1"] + df_all["VHi1"]*1j)**2
df_all["Re{I1} Rot"] = df_all["IHr1"]
df_all["Im{I1} Rot"] = df_all["IHi1"]
df_all["Re{I2} Rot"] = df_all["IHr2"]
df_all["Im{I2} Rot"] = df_all["IHi2"]
df_all["Re{A1} Rot"] = A1.apply(lambda x: x.real)
df_all["Im{A1} Rot"] = A1.apply(lambda x: x.imag)
df_all["Re{A2} Rot"] = A2.apply(lambda x: x.real)
df_all["Im{A2} Rot"] = A2.apply(lambda x: x.imag)

#Updating Units
V_I_units = [col
          for x in range(1, num_currents + 1)
          for col in ("A (I_peak, rotated)", "A (I_peak, rotated)", "V (V_peak, rotated)", "V (V_peak, rotated)")]
added_cols = [col
          for x in range(1, 3)
          for col in (f"Re{{I{x}}} Rot", f"Im{{I{x}}} Rot", f"Re{{A{x}}} Rot", f"Im{{A{x}}} Rot")]
added_units = ["A (I_peak, rotated)", "A (I_peak, rotated)", "ohm^-1 (A_1, rotated)", "ohm^-1 (A_1, rotated)"] +\
            ["A (I_peak, rotated)", "A (I_peak, rotated)", "A/V^2 (A_2, rotated)", "A/V^2 (A_2, rotated)"]
update_units(units,["theta",added_cols,V_I_cols],["Radians",added_units,V_I_units],df_all)

#Saving/Displaying Result
save_path = os.path.join(current_file_path,"rotated_data_noCdl.csv")
save_units = os.path.join(current_file_path,"rotated_data_noCdl_units.csv")
df_all.to_csv(save_path,index=False)
save_visualize_units(units, save_units, disp = False)
df_all


,Freq,Vmod,IHr1,IHi1,VHr1,VHi1,IHr2,IHi2,VHr2,VHi2,IHr3,IHi3,VHr3,VHi3,IHr4,IHi4,VHr4,VHi4,IHr5,IHi5,VHr5,VHi5,IHr6,IHi6,VHr6,VHi6,IHr7,IHi7,VHr7,VHi7,IHr8,IHi8,VHr8,VHi8,IHr9,IHi9,VHr9,VHi9,IHr10,IHi10,VHr10,VHi10,Modulation (RMS),File Number,Rank,theta,Re{I1} Rot,Im{I1} Rot,Re{I2} Rot,Im{I2} Rot,Re{A1} Rot,Im{A1} Rot,Re{A2} Rot,Im{A2} Rot
0,1.500059e+06,0.0100416,0.002713,0.001002,0.014201,-1.734723e-18,-5.960097e-06,1.137790e-06,-1.485625e-05,3.296248e-05,2.910715e-06,-7.324298e-07,3.827785e-06,-1.116321e-05,6.858451e-08,3.951056e-07,1.990658e-07,2.846193e-06,3.536293e-07,-4.379531e-07,1.344583e-06,3.765446e-07,-4.355456e-07,-2.497251e-07,-4.410651e-06,-2.638856e-06,3.371757e-07,5.533465e-07,-1.634302e-06,8.632333e-07,-2.918334e-07,-1.302901e-07,1.138403e-06,-1.963336e-06,-6.041890e-08,-8.306214e-07,-7.801283e-07,-8.252656e-07,-5.966849e-07,-2.863144e-07,-1.144894e-06,-1.740850e-06,10mV,2,73,-2.213734,0.002713,0.001002,-5.960097e-06,1.137790e-06,0.191033,0.070546,-0.029554,0.005642
1,1.191504e+06,0.0127297,0.003696,0.001032,0.018003,2.602085e-18,-1.023522e-05,1.435368e-06,-3.823949e-05,3.905072e-05,2.023492e-06,-1.592086e-06,-1.143913e-06,-8.834395e-06,6.598467e-08,5.525881e-07,6.565945e-08,2.171299e-06,1.726282e-07,-5.039619e-07,-4.873946e-06,-3.470839e-06,5.248467e-07,5.674142e-08,3.120782e-06,8.437592e-07,2.439136e-07,-5.597508e-07,-4.393330e-06,-1.479393e-06,-1.209395e-07,-1.524525e-07,2.566446e-06,-1.158303e-06,1.486426e-07,-5.202471e-08,3.870006e-06,2.865396e-06,1.656293e-07,-5.374137e-07,1.607043e-06,-4.635828e-06,10mV,2,72,2.717590,0.003696,0.001032,-1.023522e-05,1.435368e-06,0.205298,0.057347,-0.031581,0.004429
2,9.464649e+05,0.0125644,0.003802,0.000848,0.017769,2.602085e-18,-8.267328e-06,2.336178e-07,-3.432771e-05,2.065717e-05,1.746892e-07,-2.959783e-06,-1.177830e-06,-9.426601e-06,-3.050415e-07,-4.357896e-07,-8.371712e-07,-4.551628e-06,-4.605268e-08,-5.391436e-07,-3.148370e-07,-3.603242e-06,4.086252e-07,2.348381e-07,-1.508417e-07,-2.102940e-06,-6.226035e-07,-3.095557e-07,-4.611932e-06,-1.382618e-06,1.543848e-07,1.216334e-07,1.399697e-06,2.100172e-06,-1.358018e-07,1.364167e-08,-3.355972e-06,-2.098270e-06,2.903706e-08,6.158918e-07,3.731902e-06,4.033756e-06,10mV,2,71,2.913982,0.003802,0.000848,-8.267328e-06,2.336178e-07,0.213951,0.047701,-0.026185,0.000740
3,7.518164e+05,0.0125862,0.003907,0.000724,0.017800,4.336809e-19,-6.042669e-06,2.283374e-07,-2.584749e-05,9.591497e-06,3.854794e-07,-1.569547e-06,1.859230e-06,3.204955e-08,4.776985e-07,1.741774e-07,3.081516e-06,1.321300e-06,1.248871e-07,2.128497e-07,4.608791e-07,-1.714540e-06,-5.243681e-07,-1.808698e-07,-3.211583e-06,7.486711e-07,4.104034e-07,2.024826e-08,1.515043e-06,-8.529685e-07,-2.533622e-09,7.748560e-08,-6.228997e-07,-3.638576e-06,-5.627803e-07,-1.286790e-07,-2.887991e-06,4.649371e-07,-7.662843e-08,-2.049705e-08,4.100309e-06,-3.430829e-07,10mV,2,70,3.014625,0.003907,0.000724,-6.042669e-06,2.283374e-07,0.219498,0.040663,-0.019073,0.000721
4,5.972461e+05,0.012618,0.003977,0.000632,0.017845,3.469447e-18,-4.245004e-06,9.885632e-07,-1.544432e-05,6.565518e-06,1.713529e-06,-1.510375e-06,8.706940e-06,-8.616418e-06,-6.737550e-07,4.813597e-08,-1.808688e-06,3.605434e-06,-4.719546e-07,1.749625e-07,3.360209e-07,-2.384463e-06,-3.186861e-07,-3.368566e-07,-5.163890e-07,-2.777269e-06,-1.589920e-07,-1.861201e-07,8.021059e-07,8.048811e-07,3.433454e-07,-4.301110e-08,-1.138903e-07,1.905453e-06,3.417448e-07,1.168705e-07,1.256754e-06,3.493041e-06,1.408464e-07,-2.479108e-08,-6.555607e-07,2.857033e-06,10mV,2,69,2.604818,0.003977,0.000632,-4.245004e-06,9.885632e-07,0.222885,0.035434,-0.013331,0.003104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,2.381860e-01,0.0200097,0.000015,0.000014,0.028298,2.385245e-18,-2.312267e-07,-2.086944e-07,-3.955585e-08,3.244305e-07,-6.096225e-07,-5.318332e-07,-1.061530e-07,-6.33297

#### The units of `A1` (Re{A1}, Im{A1}) is A/V and `A2` (Re{A2}, Im{A2}) is A/V<sup>2</sup>.

# 4. Cdl Correction (12 Lowest Frequencies $\omega$ ONLY)

Finding the best fit Cdl to shift the I1 vectors upwards to the 45–degree line. This effect was only added to the 12 lowest frequencies.

In [9]:
def Cdl_model(xdata, Cdl):
    f, E, Iimag = xdata
    return Iimag + Cdl * f * math.pi * 2 * E

#Filtering to 12 Lowest Frequencies and Weakly Nonlinear Modulations
mod_used = ["1mV","5mV","10mV","20mV"]
df_12_freq_filter = df_all[(df_all["Rank"] <= 12) & (df_all["Modulation (RMS)"].isin(mod_used))]

# Fit
xdata = (df_12_freq_filter["Freq"],df_12_freq_filter["VHr1"], df_12_freq_filter["Im{I1} Rot"])
p0 = [1e-5]
popt, pcov = curve_fit(Cdl_model, xdata, df_12_freq_filter["Re{I1} Rot"], p0=p0) #uses a summed squared error for real and imaginary

#Display Result
Cdl_fit = popt[0]
Cdl_err = np.sqrt(np.diag(pcov))[0]
print("Best fit Cdl", Cdl_fit)
print("Standard error:", Cdl_err)

Best fit Cdl 1.2077942876489702e-05
Standard error: 2.4484945113377715e-07


In [10]:
#Correct IHi1 Using the Fit Cdl
df_12_freq = df_all[df_all["Rank"] <= 12].copy()
df_12_freq['IHi1'] = df_12_freq['IHi1'] + \
    Cdl_fit * df_12_freq["Freq"].astype(float) * 2 * math.pi * df_12_freq["VHr1"]  
    
#Add Processed Admittance and Current Magnitude Data Columns
A1 = (df_12_freq["IHr1"] + df_12_freq["IHi1"]*1j) / (df_12_freq["VHr1"] + df_12_freq["VHi1"]*1j)
A2 = (df_12_freq["IHr2"] + df_12_freq["IHi2"]*1j) / (df_12_freq["VHr1"] + df_12_freq["VHi1"]*1j)**2
df_12_freq["Re{A1} Processed"] = A1.apply(lambda x: x.real)
df_12_freq["Im{A1} Processed"] = A1.apply(lambda x: x.imag)
df_12_freq["Re{A2} Processed"] = A2.apply(lambda x: x.real)
df_12_freq["Im{A2} Processed"] = A2.apply(lambda x: x.imag)
df_12_freq["I1 Mag (A * 10^6)"] = np.sqrt(df_12_freq["IHr1"]**2 + df_12_freq["IHi1"]**2) * 10**6
df_12_freq["I2 Mag (A * 10^6)"] = np.sqrt(df_12_freq["IHr2"]**2 + df_12_freq["IHi2"]**2) * 10**6
df_12_freq["I3 Mag (A * 10^6)"] = np.sqrt(df_12_freq["IHr3"]**2 + df_12_freq["IHi3"]**2) * 10**6
df_12_freq["File Number"] = df_12_freq["File Number"].astype(int)
order = ['1mV', '5mV', '10mV', '20mV', '50mV', '100mV']  #sorting order
df_12_freq["Modulation (RMS)"] = pd.Categorical(
    df_12_freq["Modulation (RMS)"], 
    categories=order, 
    ordered=True
)
df_12_freq.sort_values(by=["Modulation (RMS)","File Number","Rank"],ascending=[True,True,True],inplace=True)
df_12_freq.reset_index(inplace=True,drop=True)

#Update Units
V_I_units = [col
          for x in range(1, num_currents + 1)
          for col in ("A (I_peak, processed)", "A (I_peak, processed)", "V (V_peak, processed)", "V (V_peak, processed)")]
added_cols = [col
          for x in range(1, 3)
          for col in (f"Re{{A{x}}} Processed", f"Im{{A{x}}} Processed")]
added_cols += [f"I{x} Mag (A * 10^6)" for x in range(1,4)]
added_units = ["ohm^-1 (A_1, Processed)"]*2 + ["A/V^2 (A_2, Processed)"]*2 + ["A (|I_peak|, Processed)"]*3
update_units(units,[V_I_cols,added_cols],[V_I_units,added_units],df_12_freq)

#Displaying Result
df_12_freq

,Freq,Vmod,IHr1,IHi1,VHr1,VHi1,IHr2,IHi2,VHr2,VHi2,IHr3,IHi3,VHr3,VHi3,IHr4,IHi4,VHr4,VHi4,IHr5,IHi5,VHr5,VHi5,IHr6,IHi6,VHr6,VHi6,IHr7,IHi7,VHr7,VHi7,IHr8,IHi8,VHr8,VHi8,IHr9,IHi9,VHr9,VHi9,IHr10,IHi10,VHr10,VHi10,Modulation (RMS),File Number,Rank,theta,Re{I1} Rot,Im{I1} Rot,Re{I2} Rot,Im{I2} Rot,Re{A1} Rot,Im{A1} Rot,Re{A2} Rot,Im{A2} Rot,Re{A1} Processed,Im{A1} Processed,Re{A2} Processed,Im{A2} Processed,I1 Mag (A * 10^6),I2 Mag (A * 10^6),I3 Mag (A * 10^6)
0,0.094659,0.0009973,5.078722e-07,4.983661e-07,0.001410,-3.049319e-20,1.225223e-10,-1.149229e-09,5.986080e-07,-4.659647e-08,1.078495e-09,2.853209e-10,3.319154e-07,8.150042e-08,-3.356929e-10,3.990471e-10,-1.488680e-08,-2.046461e-07,-9.236126e-10,-4.695625e-10,-4.725323e-07,3.602307e-08,3.756436e-11,-5.782455e-10,-2.173934e-07,1.229697e-07,6.290724e-10,1.601364e-10,2.627649e-07,-1.467205e-07,4.035190e-11,4.267114e-10,2.499666e-07,1.750087e-07,-5.512924e-10,-4.360208e-10,-4.804177e-07,-6.460902e-08,3.875099e-11,-1.538242e-10,7.372436e-08,-2.235956e-08,1mV,1,1,1.577347,5.078722e-07,4.882350e-07,1.225223e-10,-1.149229e-09,0.000360,0.000346,0.000062,-0.000578,0.000360,0.000353,0.000062,-0.000578,0.711550,0.001156,0.001116
1,0.119335,0.001,5.706766e-07,5.617096e-07,0.001414,-2.879912e-20,-1.320847e-10,-9.175286e-10,6.767822e-07,2.991005e-07,6.318670e-10,3.202578e-10,4.437005e-07,6.190784e-08,-2.637631e-10,2.010352e-10,-2.356996e-07,3.767066e-08,-7.729599e-10,-4.627487e-10,-6.521359e-07,-5.120400e-08,-1.582069e-11,-3.165148e-10,-7.793551e-08,-1.468426e-08,3.948075e-10,3.268939e-10,3.294710e-07,1.487075e-07,1.068867e-10,2.981053e-10,5.959417e-08,-1.665101e-08,-3.862077e-10,-2.862640e-10,-2.167305e-07,-7.785317e-08,1.366226e-10,-1.688850e-10,1.985113e-07,-3.434506e-08,1mV,1,2,1.581362,5.706766e-07,5.489028e-07,-1.320847e-10,-9.175286e-10,0.000404,0.000388,-0.000066,-0.000459,0.000404,0.000397,-0.000066,-0.000459,0.800743,0.000927,0.000708
2,0.150240,0.0009999,6.397478e-07,6.308609e-07,0.001414,1.355253e-19,-2.736589e-10,-4.973401e-10,3.866309e-07,6.235533e-08,7.077592e-10,6.542928e-10,6.341874e-07,-1.010781e-07,-1.132371e-10,1.190798e-10,1.324427e-07,-2.601497e-08,-7.650665e-10,-5.569686e-10,-4.830883e-07,7.263327e-08,-9.995592e-11,-2.784321e-10,5.041592e-08,-2.003982e-07,4.238345e-10,5.654478e-11,1.295478e-07,-1.623930e-07,6.857550e-11,3.267835e-10,5.969746e-09,2.691738e-08,-3.095028e-10,-2.104008e-10,-1.682510e-07,1.564113e-08,-8.456912e-11,5.487004e-11,-3.536593e-08,2.012940e-07,1mV,1,3,1.589897,6.397478e-07,6.147380e-07,-2.736589e-10,-4.973401e-10,0.000452,0.000435,-0.000137,-0.000249,0.000452,0.000446,-0.000137,-0.000249,0.898478,0.000568,0.000964
3,0.189012,0.001,7.155233e-07,7.081736e-07,0.001414,-3.388132e-20,8.797291e-11,-7.727625e-10,3.892515e-07,-2.699237e-07,7.810264e-10,5.054437e-10,6.714604e-07,4.852189e-07,-5.378187e-10,-5.162383e-11,-3.299791e-07,4.315036e-08,-7.423176e-10,-5.495371e-10,-3.827973e-07,-1.133620e-07,7.249937e-11,-2.283425e-10,-3.861552e-08,-1.095805e-07,3.914358e-10,4.185232e-10,1.298669e-07,1.592547e-07,1.495218e-10,2.843294e-10,1.369501e-07,-4.573999e-08,-3.587261e-10,-4.420293e-10,-2.747946e-07,-7.810112e-08,1.177849e-10,-1.566644e-10,-8.714233e-08,-5.847727e-08,1mV,1,4,1.601049,7.155233e-07,6.878894e-07,8.797291e-11,-7.727625e-10,0.000506,0.000486,0.000044,-0.000386,0.000506,0.000501,0.000044,-0.000386,1.006719,0.000778,0.000930
4,0.238186,0.0010009,8.020852e-07,7.966554e-07,0.001415,-1.084202e-19,1.139039e-10,-4.987984e-10,6.971743e-07,5.973938e-08,7.213182e-10,3.708663e-10,5.416546e-07,2.345523e-07,-6.067342e-10,-3.382848e-11,-2.861768e-07,-3.449831e-08,-1.060416e-09,-7.165689e-10,-4.467693e-07,-6.077237e-08,-3.332531e-10,-2.278565e-10,-3.301363e-07,3.534876e-07,4.109106e-10,1.622346e-10,2.076342e-07,-8.444250e-08,1.210564e-10,1.427429e-10,1.895208e-07,7.194417e-08,-3.787853e-10,-6.103650e-10,-2.719794e-07,-1.321633e-07,2.859411e-10,-2.114038e-10,1.236363e-07,-7.735635e-08,1mV,1,5,1.615278,8.020852e-07,7.710708e-07,1.139039e-10,-4.987984e-10,0.

#### These are the final processed IHj and VHj vectors used in the paper. `A1 Rot` represent the 2nd-order admittances prior to adding the capacitive effect, while `A1 Processed` represent the fully processed 2nd-order admittances. `A2 Rot` and `A2 Processed` are the same (since no capacitive effect is added) but were created to alleviate confusion during plotting.

# 5. Addmittance Columns for Parameter Fitting (12 Lowest Frequencies $\omega$ ONLY)

In [11]:
F = 96485 #C/mol e-
R = 8.314 #J/mol-K
T = 298 #K, assuming room temperature
f_val =  F / (R * T)

### Method 1 - Frequency Normalized Current Magnitudes

In [12]:
#Add Frequency Normalized Current Magnitudes
df_12_freq["Freq"] = df_12_freq["Freq"].astype(float)
df_12_freq["I1_mag_div_g1"] = df_12_freq["I1 Mag (A * 10^6)"] / (np.sqrt(df_12_freq["Freq"] * 2 * math.pi/2) * f_val * np.sqrt(2))
df_12_freq["I2_mag_div_g2"] = df_12_freq["I2 Mag (A * 10^6)"] / (np.sqrt(2 * df_12_freq["Freq"] * 2 * math.pi / 2) * f_val**2 * np.sqrt(2) / 8)

#Calculate h_1(∆E) and h_2(∆E) Functions
h1_2nd = [1] * len(df_12_freq)
h1_4th = 1 - df_12_freq["VHr1"]**2 * f_val**2 / 16
h1_6th = 1 - df_12_freq["VHr1"]**2 * f_val**2 / 16 + df_12_freq["VHr1"]**4 * f_val**4 / 192
h2_2nd = [1] * len(df_12_freq)
h2_4th = 1 - 7 * df_12_freq["VHr1"]**2 * f_val**2 / 24
h2_6th = 1 - 7 * df_12_freq["VHr1"]**2 * f_val**2 / 24 + 19 * df_12_freq["VHr1"]**4 * f_val**4 / 256

#Add h_1(∆E) and h_2(∆E) Columns
df_12_freq["h1_2nd"] = h1_2nd
df_12_freq["h1_4th"] = h1_4th
df_12_freq["h1_6th"] = h1_6th
df_12_freq["h2_2nd"] = h2_2nd
df_12_freq["h2_4th"] = h2_4th
df_12_freq["h2_6th"] = h2_6th

#Update Units
added_cols = ["I1_mag_div_g1","I2_mag_div_g2","h1_2nd","h1_4th","h1_6th","h2_2nd","h2_4th","h2_6th"]
added_units = ["μA*V*s^0.5","μA*V^2*s^0.5"] + ["None"]*(len(added_cols)-2)
update_units(units,added_cols,added_units,df_12_freq)

### Method 2 - Nonlinearly Corrected Admittances

In [13]:
#Add Nonlinearly Corrected Admittance Columns
h1s = [h1_2nd,h1_4th,h1_6th]
h2s = [h2_2nd,h2_4th,h2_6th]
orders = ["2nd","4th","6th"]
added_cols = []
added_units = []
for h1,h2,order in zip(h1s,h2s,orders):
    A1 = (df_12_freq["IHr1"] + df_12_freq["IHi1"]*1j) / ((df_12_freq["VHr1"] + df_12_freq["VHi1"]*1j) * h1)
    A2 = (df_12_freq["IHr2"] + df_12_freq["IHi2"]*1j) / (((df_12_freq["VHr1"] + df_12_freq["VHi1"]*1j)**2) * h2)
    df_12_freq[f"Re{{A1}}, {order} order"] = A1.apply(lambda x: x.real)
    df_12_freq[f"Im{{A1}}, {order} order"] = A1.apply(lambda x: x.imag)
    df_12_freq[f"Re{{A2}}, {order} order"] = A2.apply(lambda x: x.real)
    df_12_freq[f"Im{{A2}}, {order} order"] = A2.apply(lambda x: x.imag)
    added_units += ["ohm^-1 (A_1, Processed)"]*2 + ["A/V^2 (A_2, Processed)"]*2
    added_cols += [f"Re{{A1}}, {order} order",f"Im{{A1}}, {order} order",f"Re{{A2}}, {order} order",f"Im{{A2}}, {order} order"]

#Update Units
update_units(units,added_cols,added_units,df_12_freq)

## Final Result:

In [14]:
save_path = os.path.join(current_file_path,"preprocessed_data_final.csv")
df_12_freq.to_csv(save_path,index=False)
df_12_freq

,Freq,Vmod,IHr1,IHi1,VHr1,VHi1,IHr2,IHi2,VHr2,VHi2,IHr3,IHi3,VHr3,VHi3,IHr4,IHi4,VHr4,VHi4,IHr5,IHi5,VHr5,VHi5,IHr6,IHi6,VHr6,VHi6,IHr7,IHi7,VHr7,VHi7,IHr8,IHi8,VHr8,VHi8,IHr9,IHi9,VHr9,VHi9,IHr10,IHi10,VHr10,VHi10,Modulation (RMS),File Number,Rank,theta,Re{I1} Rot,Im{I1} Rot,Re{I2} Rot,Im{I2} Rot,Re{A1} Rot,Im{A1} Rot,Re{A2} Rot,Im{A2} Rot,Re{A1} Processed,Im{A1} Processed,Re{A2} Processed,Im{A2} Processed,I1 Mag (A * 10^6),I2 Mag (A * 10^6),I3 Mag (A * 10^6),I1_mag_div_g1,I2_mag_div_g2,h1_2nd,h1_4th,h1_6th,h2_2nd,h2_4th,h2_6th,"Re{A1}, 2nd order","Im{A1}, 2nd order","Re{A2}, 2nd order","Im{A2}, 2nd order","Re{A1}, 4th order","Im{A1}, 4th order","Re{A2}, 4th order","Im{A2}, 4th order","Re{A1}, 6th order","Im{A1}, 6th order","Re{A2}, 6th order","Im{A2}, 6th order"
0,0.094659,0.0009973,5.078722e-07,4.983661e-07,0.001410,-3.049319e-20,1.225223e-10,-1.149229e-09,5.986080e-07,-4.659647e-08,1.078495e-09,2.853209e-10,3.319154e-07,8.150042e-08,-3.356929e-10,3.990471e-10,-1.488680e-08,-2.046461e-07,-9.236126e-10,-4.695625e-10,-4.725323e-07,3.602307e-08,3.756436e-11,-5.782455e-10,-2.173934e-07,1.229697e-07,6.290724e-10,1.601364e-10,2.627649e-07,-1.467205e-07,4.035190e-11,4.267114e-10,2.499666e-07,1.750087e-07,-5.512924e-10,-4.360208e-10,-4.804177e-07,-6.460902e-08,3.875099e-11,-1.538242e-10,7.372436e-08,-2.235956e-08,1mV,1,1,1.577347,5.078722e-07,4.882350e-07,1.225223e-10,-1.149229e-09,0.000360,0.000346,0.000062,-0.000578,0.000360,0.000353,0.000062,-0.000578,0.711550,0.001156,0.001116,0.023692,0.000006,1,0.999811,0.999812,1,0.999120,0.999121,0.000360,0.000353,0.000062,-0.000578,0.000360,0.000353,0.000062,-0.000578,0.000360,0.000353,6.165290e-05,-5.782892e-04
1,0.119335,0.001,5.706766e-07,5.617096e-07,0.001414,-2.879912e-20,-1.320847e-10,-9.175286e-10,6.767822e-07,2.991005e-07,6.318670e-10,3.202578e-10,4.437005e-07,6.190784e-08,-2.637631e-10,2.010352e-10,-2.356996e-07,3.767066e-08,-7.729599e-10,-4.627487e-10,-6.521359e-07,-5.120400e-08,-1.582069e-11,-3.165148e-10,-7.793551e-08,-1.468426e-08,3.948075e-10,3.268939e-10,3.294710e-07,1.487075e-07,1.068867e-10,2.981053e-10,5.959417e-08,-1.665101e-08,-3.862077e-10,-2.862640e-10,-2.167305e-07,-7.785317e-08,1.366226e-10,-1.688850e-10,1.985113e-07,-3.434506e-08,1mV,1,2,1.581362,5.706766e-07,5.489028e-07,-1.320847e-10,-9.175286e-10,0.000404,0.000388,-0.000066,-0.000459,0.000404,0.000397,-0.000066,-0.000459,0.800743,0.000927,0.000708,0.023746,0.000004,1,0.999810,0.999810,1,0.999115,0.999116,0.000404,0.000397,-0.000066,-0.000459,0.000404,0.000397,-0.000066,-0.000459,0.000404,0.000397,-6.610590e-05,-4.592059e-04
2,0.150240,0.0009999,6.397478e-07,6.308609e-07,0.001414,1.355253e-19,-2.736589e-10,-4.973401e-10,3.866309e-07,6.235533e-08,7.077592e-10,6.542928e-10,6.341874e-07,-1.010781e-07,-1.132371e-10,1.190798e-10,1.324427e-07,-2.601497e-08,-7.650665e-10,-5.569686e-10,-4.830883e-07,7.263327e-08,-9.995592e-11,-2.784321e-10,5.041592e-08,-2.003982e-07,4.238345e-10,5.654478e-11,1.295478e-07,-1.623930e-07,6.857550e-11,3.267835e-10,5.969746e-09,2.691738e-08,-3.095028e-10,-2.104008e-10,-1.682510e-07,1.564113e-08,-8.456912e-11,5.487004e-11,-3.536593e-08,2.012940e-07,1mV,1,3,1.589897,6.397478e-07,6.147380e-07,-2.736589e-10,-4.973401e-10,0.000452,0.000435,-0.000137,-0.000249,0.000452,0.000446,-0.000137,-0.000249,0.898478,0.000568,0.000964,0.023746,0.000002,1,0.999810,0.999811,1,0.999115,0.999116,0.000452,0.000446,-0.000137,-0.000249,0.000452,0.000446,-0.000137,-0.000249,0.000452,0.000446,-1.369701e-04,-2.489256e-04
3,0.189012,0.001,7.155233e-07,7.081736e-07,0.001414,-3.388132e-20,8.797291e-11,-7.727625e-10,3.892515e-07,-2.699237e-07,7.810264e-10,5.054437e-10,6.714604e-07,4.852189e-07,-5.378187e-10,-5.162383e-11,-3.299791e-07,4.315036e-08,-7.423176e-10,-5.495371e-10,-3.827973e-07,-1.133620e-07,7.249937e-11,-2.283425e-10,-3.861552e-08,-1.095805e-07,3.914358e-10,4.185232e-10,1.298669e-07,1.592547e-07,1.495218e-10,2.843294e-10,1.369501e-07,-4.573999e-08,-3.587261e-10,-4.420293e-10,-2.747946e-07,-7.810112e-08,1.177849

In [15]:
#Update / Display Units
save_units = os.path.join(current_file_path,"preprocessed_data_final_units.csv")
save_visualize_units(units, save_units, disp = True)

,Unit,Value
0,Freq,Hz
1,Vmod,V
2,IHr1,"A (I_peak, processed)"
3,IHi1,"A (I_peak, processed)"
4,VHr1,"V (V_peak, processed)"
5,VHi1,"V (V_peak, processed)"
6,IHr2,"A (I_peak, processed)"
7,IHi2,"A (I_peak, processed)"
8,VHr2,"V (V_peak, processed)"
9,VHi2,"V (V_peak, processed)"
